# 2. Information Protection and DLP

## Data classification

Before you can protect data, you need to know *what kind* of data you have. Microsoft Purview classifies data using:

### Sensitive information types (SIT)

Pattern-based classifiers that detect specific data formats:

| SIT | Pattern | Example match |
|-----|---------|---------------|
| Credit card number | 16 digits with Luhn checksum | 4111-1111-1111-1111 |
| SSN (US) | XXX-XX-XXXX pattern | 123-45-6789 |
| Email address | user@domain pattern | john@contoso.com |
| Passport number | Country-specific patterns | Various |
| IBAN | Country code + check digits + account | DE89 3704 0044 0532 0130 00 |

### Trainable classifiers

ML models trained to recognize *types of content* (not just patterns):
- Resumes/CVs
- Source code
- Harassment language
- Financial statements

### Content and Activity Explorer

- **Content Explorer**: browse what sensitive data exists across M365 (SharePoint, OneDrive, Exchange).
- **Activity Explorer**: see what users are doing with sensitive data (downloaded, shared, labeled).

Let's build a sensitive information detector:

In [ ]:
import re

# Simulate Microsoft Purview sensitive information type detection
SENSITIVE_INFO_TYPES = [
    {
        'name': 'Credit Card Number',
        'pattern': r'\b(?:4[0-9]{12}(?:[0-9]{3})?|5[1-5][0-9]{14}|3[47][0-9]{13})\b',
        'validator': lambda m: _luhn_check(re.sub(r'[\s-]', '', m)),
        'confidence': 'high',
    },
    {
        'name': 'US Social Security Number',
        'pattern': r'\b\d{3}-\d{2}-\d{4}\b',
        'validator': lambda m: not m.startswith('000') and not m.startswith('666'),
        'confidence': 'high',
    },
    {
        'name': 'Email Address',
        'pattern': r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
        'validator': lambda m: True,
        'confidence': 'medium',
    },
    {
        'name': 'IP Address',
        'pattern': r'\b(?:(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\.){3}(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\b',
        'validator': lambda m: True,
        'confidence': 'low',
    },
]

def _luhn_check(num: str) -> bool:
    digits = [int(d) for d in num if d.isdigit()]
    odd_digits = digits[-1::-2]
    even_digits = digits[-2::-2]
    total = sum(odd_digits)
    for d in even_digits:
        total += sum(divmod(d * 2, 10))
    return total % 10 == 0

def scan_for_sensitive_data(text: str) -> list:
    findings = []
    for sit in SENSITIVE_INFO_TYPES:
        matches = re.finditer(sit['pattern'], text)
        for match in matches:
            value = match.group()
            if sit['validator'](value):
                findings.append({
                    'type': sit['name'],
                    'value': value[:4] + '***' + value[-4:] if len(value) > 8 else '***',
                    'position': match.start(),
                    'confidence': sit['confidence'],
                })
    return findings

# Scan sample documents
DOCUMENTS = [
    {
        'name': 'employee_records.xlsx',
        'content': 'Employee: John Doe, SSN: 123-45-6789, Email: john.doe@contoso.com, Phone: 555-0123',
    },
    {
        'name': 'payment_log.csv',
        'content': 'Transaction: card 4111111111111111, amount $500. Merchant IP: 192.168.1.100.',
    },
    {
        'name': 'meeting_notes.docx',
        'content': 'Discussed Q3 projections. Alice mentioned the project is on track. No action items.',
    },
]

print('=== Microsoft Purview: Sensitive Information Scan ===\n')
for doc in DOCUMENTS:
    findings = scan_for_sensitive_data(doc['content'])
    icon = '🔴' if findings else '🟢'
    print(f'{icon} {doc["name"]}')
    if findings:
        for f in findings:
            print(f'   ⚠️  {f["type"]} detected (confidence: {f["confidence"]}): {f["value"]}')
    else:
        print(f'   No sensitive data found')
    print()

---
## Sensitivity Labels

Once you know data is sensitive, you **label** it. Labels travel with the document and enforce protection:

| Label | What it does | Example |
|-------|-------------|----------|
| **Public** | No restrictions | Press releases |
| **General** | No protection but marked | Internal memos |
| **Confidential** | Encryption + access control | Financial reports |
| **Highly Confidential** | Encryption + watermark + no forwarding | M&A documents, PII |

Labels can:
- **Encrypt** content (Azure RMS)
- **Add watermarks** (header/footer/watermark)
- **Restrict access** (only specific users/groups)
- **Auto-apply** when sensitive data is detected

### Label policies

Policies control which labels are available to which users and set defaults:
- Publish labels to specific groups
- Set a default label (e.g., "General" for all new docs)
- Require justification when downgrading (Confidential → Public)

---
## Data Loss Prevention (DLP)

DLP prevents users from accidentally (or intentionally) sharing sensitive data. It works across:
- Exchange email
- SharePoint/OneDrive
- Teams chat and channels
- Endpoint devices (Windows/macOS)

A DLP policy has:
1. **Conditions** — what triggers the policy (e.g., document contains credit card numbers)
2. **Actions** — what happens (block sharing, notify user, alert admin)
3. **User notifications** — policy tips that educate users

In [ ]:
# Simulate DLP policy evaluation
DLP_POLICIES = [
    {
        'name': 'Block external sharing of PII',
        'conditions': {'sensitive_types': ['US Social Security Number', 'Credit Card Number'], 'min_count': 1},
        'actions': {'internal_share': 'allow_with_warning', 'external_share': 'block', 'email_external': 'block'},
    },
    {
        'name': 'Warn on bulk PII sharing',
        'conditions': {'sensitive_types': ['US Social Security Number'], 'min_count': 5},
        'actions': {'internal_share': 'block', 'external_share': 'block', 'email_external': 'block'},
    },
]

def evaluate_dlp(document_findings: list, action_type: str) -> list:
    results = []
    finding_types = [f['type'] for f in document_findings]
    for policy in DLP_POLICIES:
        matching = [t for t in policy['conditions']['sensitive_types'] if t in finding_types]
        count = len([f for f in document_findings if f['type'] in matching])
        if count >= policy['conditions']['min_count']:
            action = policy['actions'].get(action_type, 'allow')
            results.append({'policy': policy['name'], 'action': action, 'matched_types': matching, 'count': count})
    return results

# Test DLP scenarios
doc_with_ssn = [{'type': 'US Social Security Number', 'value': '123-***-6789', 'confidence': 'high'}]
doc_with_card = [{'type': 'Credit Card Number', 'value': '4111***1111', 'confidence': 'high'}]
doc_clean = []

scenarios = [
    ('Share SSN doc externally', doc_with_ssn, 'external_share'),
    ('Share SSN doc internally', doc_with_ssn, 'internal_share'),
    ('Email credit card doc to external', doc_with_card, 'email_external'),
    ('Share clean doc externally', doc_clean, 'external_share'),
]

print('=== DLP Policy Evaluation ===\n')
for name, findings, action_type in scenarios:
    results = evaluate_dlp(findings, action_type)
    if not results:
        print(f'✅ {name}: ALLOWED (no DLP policy triggered)')
    else:
        for r in results:
            icons = {'allow': '✅', 'allow_with_warning': '⚠️', 'block': '🚫'}
            print(f'{icons[r["action"]]} {name}: {r["action"].upper()} — policy: "{r["policy"]}"')
    print()

---
## Records Management and Retention

### Retention policies

Control how long content is **kept** and when it's **deleted**:

| Setting | Options |
|---------|----------|
| **Retain for** | X days/months/years |
| **After retention** | Delete automatically, or do nothing |
| **Apply to** | Exchange, SharePoint, OneDrive, Teams, Yammer |
| **Scope** | All users, specific users/groups, specific sites |

### Retention labels

More granular than policies — applied to individual items (emails, documents):
- Can be applied **manually** by users or **automatically** by rules
- Can mark items as **records** (can't be modified or deleted)
- Can mark items as **regulatory records** (can't even be removed by admins)

### Key rule: retention wins over deletion

If a retention policy says "keep for 7 years" and a user deletes the file, it's preserved in a hidden location for 7 years.

**Exam tip**: retention policies are applied at the *location* level (all of SharePoint), while retention labels are applied at the *item* level (specific document).

---
## Summary

| Concept | Key fact |
|---------|----------|
| **Sensitive info types** | Pattern-based detection (SSN, credit cards, etc.) |
| **Trainable classifiers** | ML-based content classification |
| **Content Explorer** | See what sensitive data exists |
| **Activity Explorer** | See what users do with sensitive data |
| **Sensitivity labels** | Classify + protect (encrypt, watermark, restrict) |
| **DLP** | Prevent sharing of sensitive data (block, warn, notify) |
| **Retention policies** | Keep/delete content by age, applied at location level |
| **Retention labels** | Keep/delete individual items, can make records |

**Next**: [Notebook 3 — Insider Risk and eDiscovery](03_insider_risk_and_ediscovery.ipynb)